In [10]:
import pandas as pd
import random
import json
from pathlib import Path

random.seed(42)

data_path = Path("../data/cases")
data_path.mkdir(parents=True, exist_ok=True)

In [11]:
def generate_ecommerce_case(i):
    price = random.choice([50, 120, 300, 600, 980, 1500])
    refunds = random.randint(0, 6)
    account_age = random.randint(5, 365)
    delivery_days = random.randint(1, 10)

    evidence = random.choice([True, False])

    # rule
    if price > 500 and refunds >= 3:
        decision = "escalate"
    elif refunds >= 5:
        decision = "reject"
    elif not evidence:
        decision = "escalate"
    else:
        decision = "approve"

    return {
        "case_id": f"E-{i:03d}",
        "domain": "ecommerce",
        "customer_profile": {
            "account_age_days": account_age,
            "num_past_refunds_60d": refunds,
        },
        "transaction": {
            "item_price": price,
            "delivery_days_ago": delivery_days
        },
        "customer_claim": random.choice([
            "item is defective",
            "wrong item received",
            "no longer needed",
            "item damaged on arrival"
        ]),
        "evidence_provided": evidence,
        "ground_truth_decision": decision
    }

In [12]:
def generate_finance_case(i):
    amount = random.choice([200, 800, 2000, 5000, 12000])
    account_age = random.randint(1, 365)
    rapid_txn = random.choice([True, False])
    geo_risk = random.choice([True, False])

    if amount > 5000 and account_age < 30:
        decision = "escalate"
    elif rapid_txn and geo_risk:
        decision = "reject"
    elif geo_risk:
        decision = "escalate"
    else:
        decision = "approve"

    return {
        "case_id": f"F-{i:03d}",
        "domain": "finance",
        "customer_profile": {
            "account_age_days": account_age
        },
        "transaction": {
            "amount": amount,
            "rapid_transactions": rapid_txn,
            "geo_risk": geo_risk
        },
        "customer_claim": "Transaction flagged for review",
        "ground_truth_decision": decision
    }

In [13]:
cases = []

# e-com
for i in range(100):
    cases.append(generate_ecommerce_case(i))

# fin
for i in range(100):
    cases.append(generate_finance_case(i))

df = pd.DataFrame(cases)
df.head()

,case_id,domain,customer_profile,transaction,customer_claim,evidence_provided,ground_truth_decision
0,E-000,ecommerce,"{'account_age_days': 17, 'num_past_refunds_60d...","{'item_price': 1500, 'delivery_days_ago': 5}",wrong item received,True,approve
1,E-001,ecommerce,"{'account_age_days': 57, 'num_past_refunds_60d...","{'item_price': 120, 'delivery_days_ago': 9}",item damaged on arrival,True,reject
2,E-002,ecommerce,"{'account_age_days': 52, 'num_past_refunds_60d...","{'item_price': 50, 'delivery_days_ago': 4}",item is defective,True,approve
3,E-003,ecommerce,"{'account_age_days': 337, 'num_past_refunds_60...","{'item_price': 980, 'delivery_days_ago': 9}",wrong item received,False,escalate
4,E-004,ecommerce,"{'account_age_days': 147, 'num_past_refunds_60...","{'item_price': 600, 'delivery_days_ago': 1}",item damaged on arrival,True,escalate


In [14]:
output_path = data_path / "cases.json"

with open(output_path, "w") as f:
    json.dump(cases, f, indent=2)

print(f"Saved to {output_path}")

Saved to ../data/cases/cases.json


In [15]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
import pandas as pd
from pathlib import Path

env_path = Path().resolve().parent / ".env"
load_dotenv(env_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")
print(api_key[:12])

client = OpenAI(api_key=api_key)

sk-proj-u4m7


In [16]:
#policy
policy_dir = Path("../data/policies")

with open(policy_dir / "ecommerce_return_policy.txt", "r") as f:
    ecommerce_policy = f.read()

with open(policy_dir / "financial_risk_policy.txt", "r") as f:
    financial_policy = f.read()

print(ecommerce_policy[:300])
print(financial_policy[:300])

In [17]:
#read cases
case_path = Path("../data/cases/cases.json")

with open(case_path, "r") as f:
    cases = json.load(f)

len(cases), cases[0]

(200,
 {'case_id': 'E-000',
  'domain': 'ecommerce',
  'customer_profile': {'account_age_days': 17, 'num_past_refunds_60d': 0},
  'transaction': {'item_price': 1500, 'delivery_days_ago': 5},
  'customer_claim': 'wrong item received',
  'evidence_provided': True,
  'ground_truth_decision': 'approve'})

In [18]:
# Design cases
def get_policy_for_case(case):
    if case["domain"] == "ecommerce":
        return ecommerce_policy
    elif case["domain"] == "finance":
        return financial_policy
    else:
        return ecommerce_policy + "\n\n" + financial_policy


def decision_agent(case):
    policy = get_policy_for_case(case)
    
    prompt = f"""
You are an AI risk decision copilot for digital transactions.

Your task is to review a customer case and make a decision based ONLY on the provided policy.

Possible decisions:
- approve
- reject
- escalate

Return your answer as valid JSON with this exact schema:
{{
  "decision": "approve/reject/escalate",
  "risk_score": number between 0 and 1,
  "risk_signals": ["signal 1", "signal 2"],
  "policy_evidence": ["specific policy rule used"],
  "reasoning": "brief explanation",
  "recommended_action": "next step for the analyst"
}}

Policy:
{policy}

Case:
{json.dumps(case, indent=2)}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a precise risk analyst. Always return valid JSON only."},
            {"role": "user", "content": prompt}
        ]
    )
    
    content = response.choices[0].message.content
    
    try:
        return json.loads(content)
    except:
        return {"error": "JSON parsing failed", "raw_output": content}

In [19]:
#case test
test_case = cases[0]
test_case


{'case_id': 'E-000',
 'domain': 'ecommerce',
 'customer_profile': {'account_age_days': 17, 'num_past_refunds_60d': 0},
 'transaction': {'item_price': 1500, 'delivery_days_ago': 5},
 'customer_claim': 'wrong item received',
 'evidence_provided': True,
 'ground_truth_decision': 'approve'}

In [20]:
result = decision_agent(test_case)
result

{'decision': 'approve',
 'risk_score': 0.2,
 'risk_signals': [],
 'policy_evidence': ['Account age is less than 30 days but no past refunds in the last 60 days.'],
 'reasoning': 'The customer has a new account but has not requested any refunds in the past 60 days, indicating low risk despite the claim.',
 'recommended_action': 'Process the refund for the customer.'}

In [21]:
results = []

for case in cases[:10]:
    ai_result = decision_agent(case)
    
    results.append({
        "case_id": case["case_id"],
        "domain": case["domain"],
        "ground_truth_decision": case["ground_truth_decision"],
        "ai_decision": ai_result.get("decision"),
        "risk_score": ai_result.get("risk_score"),
        "reasoning": ai_result.get("reasoning"),
        "raw_result": ai_result
    })

results_df = pd.DataFrame(results)
results_df

,case_id,domain,ground_truth_decision,ai_decision,risk_score,reasoning,raw_result
0,E-000,ecommerce,approve,approve,0.20,The customer has a new account but has not req...,"{'decision': 'approve', 'risk_score': 0.2, 'ri..."
1,E-001,ecommerce,reject,reject,0.75,The customer has a high number of past refunds...,"{'decision': 'reject', 'risk_score': 0.75, 'ri..."
2,E-002,ecommerce,approve,approve,0.10,The customer has a relatively new account but ...,"{'decision': 'approve', 'risk_score': 0.1, 'ri..."
3,E-003,ecommerce,escalate,escalate,0.70,The customer has reported receiving the wrong ...,"{'decision': 'escalate', 'risk_score': 0.7, 'r..."
4,E-004,ecommerce,escalate,escalate,0.70,The customer has a high number of past refunds...,"{'decision': 'escalate', 'risk_score': 0.7, 'r..."
5,E-005,ecommerce,escalate,escalate,0.70,The customer has a relatively short account ag...,"{'decision': 'escalate', 'risk_score': 0.7, 'r..."
6,E-006,ecommerce,escalate,escalate,0.70,The customer has a relatively short account ag...,"{'decision': 'escalate', 'risk_score': 0.7, 'r..."
7,E-007,ecommerce,reject,reject,0.75,The customer has a high number of past refunds...,"{'decision': 'reject', 'risk_score': 0.75, 'ri..."
8,E-008,ecommerce,escalate,escalate,0.70,The customer has a high number of past refunds...,"{'decision': 'escalate', 'risk_score': 0.7, 'r..."
9,E-009,ecommerce,escalate,escalate,0.70,The customer is relatively new with an account...,"{'decision': 'escalate', 'risk_score': 0.7, 'r..."


In [22]:
results_df["match"] = results_df["ground_truth_decision"] == results_df["ai_decision"]
accuracy = results_df["match"].mean()

print(f"Decision accuracy on first 10 cases: {accuracy:.2%}")
results_df[["case_id", "domain", "ground_truth_decision", "ai_decision", "risk_score", "match"]]

Decision accuracy on first 10 cases: 100.00%


,case_id,domain,ground_truth_decision,ai_decision,risk_score,match
0,E-000,ecommerce,approve,approve,0.20,True
1,E-001,ecommerce,reject,reject,0.75,True
2,E-002,ecommerce,approve,approve,0.10,True
3,E-003,ecommerce,escalate,escalate,0.70,True
4,E-004,ecommerce,escalate,escalate,0.70,True
5,E-005,ecommerce,escalate,escalate,0.70,True
6,E-006,ecommerce,escalate,escalate,0.70,True
7,E-007,ecommerce,reject,reject,0.75,True
8,E-008,ecommerce,escalate,escalate,0.70,True
9,E-009,ecommerce,escalate,escalate,0.70,True


In [23]:
#judge evaluator
def llm_judge(case, ai_result):
    policy = get_policy_for_case(case)

    judge_prompt = f"""
You are an independent evaluator for an AI risk decision system.

Evaluate whether the AI decision is correct, policy-grounded, and useful for a human analyst.

Return valid JSON only with this schema:
{{
  "decision_correctness": 0-10,
  "policy_grounding": 0-10,
  "reasoning_quality": 0-10,
  "risk_score_alignment": 0-10,
  "overall_score": 0-10,
  "main_issue": "briefly describe the biggest issue, or 'none'",
  "improvement_suggestion": "brief suggestion"
}}

Ground Truth Decision:
{case["ground_truth_decision"]}

Policy:
{policy}

Case:
{json.dumps(case, indent=2)}

AI Result:
{json.dumps(ai_result, indent=2)}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a strict evaluator. Return valid JSON only."},
            {"role": "user", "content": judge_prompt}
        ]
    )

    content = response.choices[0].message.content

    try:
        return json.loads(content)
    except:
        return {"error": "judge json parsing failed", "raw_output": content}

In [24]:
judge_results = []

for _, row in results_df.iterrows():
    judge = llm_judge(row["raw_result"]["case"] if "case" in row["raw_result"] else cases[int(row.name)], row["raw_result"])

    judge_results.append({
        "case_id": row["case_id"],
        "domain": row["domain"],
        "ground_truth_decision": row["ground_truth_decision"],
        "ai_decision": row["ai_decision"],
        "rule_match": row["match"],
        "risk_score": row["risk_score"],
        "judge_overall": judge.get("overall_score"),
        "decision_correctness": judge.get("decision_correctness"),
        "policy_grounding": judge.get("policy_grounding"),
        "reasoning_quality": judge.get("reasoning_quality"),
        "risk_score_alignment": judge.get("risk_score_alignment"),
        "main_issue": judge.get("main_issue"),
        "improvement_suggestion": judge.get("improvement_suggestion"),
        "raw_judge": judge
    })

judge_df = pd.DataFrame(judge_results)
judge_df

,case_id,domain,ground_truth_decision,ai_decision,rule_match,risk_score,judge_overall,decision_correctness,policy_grounding,reasoning_quality,risk_score_alignment,main_issue,improvement_suggestion,raw_judge
0,E-000,ecommerce,approve,approve,True,0.20,8,10,8,9,7,Risk signals from a new account and high item ...,Consider implementing additional verification ...,"{'decision_correctness': 10, 'policy_grounding..."
1,E-001,ecommerce,reject,reject,True,0.75,9,10,9,8,9,none,Provide more detailed reasoning on how the ris...,"{'decision_correctness': 10, 'policy_grounding..."
2,E-002,ecommerce,approve,approve,True,0.10,9,10,9,9,10,none,Ensure clarity in the reasoning by explicitly ...,"{'decision_correctness': 10, 'policy_grounding..."
3,E-003,ecommerce,escalate,escalate,True,0.70,9,10,10,9,9,none,Provide more detailed reasoning for the decision,"{'decision_correctness': 10, 'policy_grounding..."
4,E-004,ecommerce,escalate,escalate,True,0.70,9,10,10,9,9,none,Provide more detailed reasoning on how the ris...,"{'decision_correctness': 10, 'policy_grounding..."
5,E-005,ecommerce,escalate,escalate,True,0.70,9,10,10,9,8,none,Provide more detailed reasoning on the implica...,"{'decision_correctness': 10, 'policy_grounding..."
6,E-006,ecommerce,escalate,escalate,True,0.70,9,10,10,9,9,none,Provide more detailed guidelines on acceptable...,"{'decision_correctness': 10, 'policy_grounding..."
7,E-007,ecommerce,reject,reject,True,0.75,9,10,9,8,9,none,Provide more detailed reasoning on how the evi...,"{'decision_correctness': 10, 'policy_grounding..."
8,E-008,ecommerce,escalate,escalate,True,0.70,9,10,10,9,8,none,Provide more detailed reasoning on how the ris...,"{'decision_correctness': 10, 'policy_grounding..."
9,E-009,ecommerce,escalate,escalate,True,0.70,9,10,10,9,9,none,Provide more detailed reasoning on the potenti...,"{'decision_correctness': 10, 'policy_grounding..."


In [25]:
score_cols = [
    "judge_overall",
    "decision_correctness",
    "policy_grounding",
    "reasoning_quality",
    "risk_score_alignment"
]

judge_df[score_cols].mean()

judge_overall            8.9
decision_correctness    10.0
policy_grounding         9.5
reasoning_quality        8.8
risk_score_alignment     8.7
dtype: float64

In [26]:
judge_df.sort_values("judge_overall").head(5)[[
    "case_id",
    "domain",
    "ground_truth_decision",
    "ai_decision",
    "risk_score",
    "judge_overall",
    "main_issue",
    "improvement_suggestion"
]]

,case_id,domain,ground_truth_decision,ai_decision,risk_score,judge_overall,main_issue,improvement_suggestion
0,E-000,ecommerce,approve,approve,0.20,8,Risk signals from a new account and high item ...,Consider implementing additional verification ...
1,E-001,ecommerce,reject,reject,0.75,9,none,Provide more detailed reasoning on how the ris...
2,E-002,ecommerce,approve,approve,0.10,9,none,Ensure clarity in the reasoning by explicitly ...
3,E-003,ecommerce,escalate,escalate,0.70,9,none,Provide more detailed reasoning for the decision
4,E-004,ecommerce,escalate,escalate,0.70,9,none,Provide more detailed reasoning on how the ris...


## hard cases

In [27]:
hard_cases = [
    {
        "case_id": "H-001",
        "domain": "ecommerce",
        "customer_profile": {
            "account_age_days": 20,
            "num_past_refunds_60d": 2
        },
        "transaction": {
            "item_price": 1200,
            "delivery_days_ago": 2
        },
        "customer_claim": "item defective",
        "evidence_provided": True,
        "ground_truth_decision": "escalate"
    },
    {
        "case_id": "H-002",
        "domain": "ecommerce",
        "customer_profile": {
            "account_age_days": 200,
            "num_past_refunds_60d": 4
        },
        "transaction": {
            "item_price": 80,
            "delivery_days_ago": 5
        },
        "customer_claim": "no longer needed",
        "evidence_provided": False,
        "ground_truth_decision": "reject"
    },
    {
        "case_id": "H-003",
        "domain": "finance",
        "customer_profile": {
            "account_age_days": 10
        },
        "transaction": {
            "amount": 6000,
            "rapid_transactions": False,
            "geo_risk": False
        },
        "customer_claim": "normal transaction",
        "ground_truth_decision": "escalate"
    }
]

In [28]:
#agent+judge
hard_results = []

for case in hard_cases:
    ai_result = decision_agent(case)
    judge = llm_judge(case, ai_result)

    hard_results.append({
        "case_id": case["case_id"],
        "gt": case["ground_truth_decision"],
        "ai": ai_result.get("decision"),
        "judge_score": judge.get("overall_score"),
        "issue": judge.get("main_issue"),
        "suggestion": judge.get("improvement_suggestion")
    })

pd.DataFrame(hard_results)

,case_id,gt,ai,judge_score,issue,suggestion
0,H-001,escalate,escalate,9,none,Provide more detailed reasoning on how the evi...
1,H-002,reject,reject,9,none,Consider providing more detailed guidance on a...
2,H-003,escalate,escalate,9,none,Consider providing more context on the custome...


In [29]:
#consistency_test
def consistency_test(case, runs=3):
    decisions = []
    scores = []

    for _ in range(runs):
        result = decision_agent(case)
        decisions.append(result.get("decision"))
        scores.append(result.get("risk_score"))

    return {
        "case_id": case["case_id"],
        "decisions": decisions,
        "scores": scores,
        "decision_consistent": len(set(decisions)) == 1,
        "score_variance": max(scores) - min(scores)
    }

In [30]:
temperature=0.3

In [31]:
consistency_results = []

for case in cases[:10]:
    consistency_results.append(consistency_test(case, runs=5))

pd.DataFrame(consistency_results)

,case_id,decisions,scores,decision_consistent,score_variance
0,E-000,"[approve, approve, approve, approve, approve]","[0.1, 0.2, 0.2, 0.2, 0.1]",True,0.10
1,E-001,"[reject, reject, reject, reject, reject]","[0.8, 0.8, 0.75, 0.8, 0.8]",True,0.05
2,E-002,"[approve, approve, approve, approve, approve]","[0.1, 0.1, 0.1, 0.1, 0.1]",True,0.00
3,E-003,"[escalate, escalate, escalate, escalate, escal...","[0.7, 0.7, 0.7, 0.7, 0.7]",True,0.00
4,E-004,"[escalate, escalate, escalate, escalate, escal...","[0.7, 0.7, 0.7, 0.7, 0.7]",True,0.00
5,E-005,"[escalate, escalate, escalate, escalate, escal...","[0.7, 0.7, 0.7, 0.7, 0.7]",True,0.00
6,E-006,"[escalate, escalate, escalate, escalate, escal...","[0.7, 0.7, 0.7, 0.7, 0.7]",True,0.00
7,E-007,"[reject, reject, reject, reject, reject]","[0.75, 0.75, 0.75, 0.75, 0.75]",True,0.00
8,E-008,"[escalate, escalate, escalate, escalate, escal...","[0.7, 0.7, 0.7, 0.7, 0.7]",True,0.00
9,E-009,"[escalate, escalate, escalate, escalate, escal...","[0.7, 0.7, 0.7, 0.7, 0.7]",True,0.00


In [32]:
borderline_cases = [
    {
        "case_id": "B-001",
        "domain": "ecommerce",
        "customer_profile": {"account_age_days": 35, "num_past_refunds_60d": 3},
        "transaction": {"item_price": 510, "delivery_days_ago": 8},
        "customer_claim": "item quality is not as expected",
        "evidence_provided": False,
        "ground_truth_decision": "escalate"
    },
    {
        "case_id": "B-002",
        "domain": "ecommerce",
        "customer_profile": {"account_age_days": 180, "num_past_refunds_60d": 3},
        "transaction": {"item_price": 120, "delivery_days_ago": 29},
        "customer_claim": "no longer needed",
        "evidence_provided": True,
        "ground_truth_decision": "escalate"
    },
    {
        "case_id": "B-003",
        "domain": "finance",
        "customer_profile": {"account_age_days": 31},
        "transaction": {"amount": 5200, "rapid_transactions": False, "geo_risk": False},
        "customer_claim": "normal business transfer",
        "ground_truth_decision": "escalate"
    },
    {
        "case_id": "B-004",
        "domain": "finance",
        "customer_profile": {"account_age_days": 120},
        "transaction": {"amount": 4800, "rapid_transactions": True, "geo_risk": False},
        "customer_claim": "urgent payment to vendor",
        "ground_truth_decision": "escalate"
    }
]

In [33]:
borderline_consistency = []

for case in borderline_cases:
    borderline_consistency.append(consistency_test(case, runs=5))

pd.DataFrame(borderline_consistency)

,case_id,decisions,scores,decision_consistent,score_variance
0,B-001,"[escalate, escalate, escalate, escalate, escal...","[0.7, 0.7, 0.7, 0.7, 0.7]",True,0.0
1,B-002,"[escalate, escalate, escalate, escalate, escal...","[0.7, 0.7, 0.7, 0.7, 0.7]",True,0.0
2,B-003,"[escalate, escalate, escalate, escalate, escal...","[0.7, 0.7, 0.7, 0.7, 0.7]",True,0.0
3,B-004,"[escalate, escalate, escalate, escalate, escal...","[0.7, 0.7, 0.7, 0.7, 0.7]",True,0.0


The consistency test showed that final decisions remained stable across 5 repeated runs, while risk scores showed minor variance in borderline cases. This suggests that the decision labels are robust under moderate stochasticity, but confidence/risk calibration still needs monitoring.

In [34]:
#Stabilization Strategy
from collections import Counter

def stable_decision(case, runs=5):
    decisions = []
    scores = []

    for _ in range(runs):
        result = decision_agent(case)
        decisions.append(result.get("decision"))
        scores.append(result.get("risk_score"))

    final_decision = Counter(decisions).most_common(1)[0][0]
    avg_score = sum(scores) / len(scores)

    return {
        "case_id": case["case_id"],
        "final_decision": final_decision,
        "avg_score": avg_score,
        "all_decisions": decisions
    }

In [35]:
#Embedding + Semantic Evaluator
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

In [36]:
def build_reference_reasoning(case):
    if case["domain"] == "ecommerce":
        profile = case["customer_profile"]
        txn = case["transaction"]
        return f"""
        Decision should be {case['ground_truth_decision']} based on ecommerce return policy.
        Item price: {txn['item_price']}.
        Past refunds in 60 days: {profile['num_past_refunds_60d']}.
        Evidence provided: {case['evidence_provided']}.
        Account age: {profile['account_age_days']} days.
        Key policy factors: high-value item, frequent refunds, evidence, manual review.
        """

    if case["domain"] == "finance":
        profile = case["customer_profile"]
        txn = case["transaction"]
        return f"""
        Decision should be {case['ground_truth_decision']} based on financial risk policy.
        Transaction amount: {txn['amount']}.
        Account age: {profile['account_age_days']} days.
        Rapid transactions: {txn['rapid_transactions']}.
        Geographic risk: {txn['geo_risk']}.
        Key policy factors: transaction size, new account risk, rapid transfers, geographic risk.
        """

In [37]:
def semantic_evaluator(case, ai_result):
    ai_reasoning = ai_result.get("reasoning", "")
    reference_reasoning = build_reference_reasoning(case)

    ai_emb = get_embedding(ai_reasoning)
    ref_emb = get_embedding(reference_reasoning)

    similarity = cosine_similarity(
        np.array(ai_emb).reshape(1, -1),
        np.array(ref_emb).reshape(1, -1)
    )[0][0]

    return {
        "case_id": case["case_id"],
        "semantic_similarity": similarity,
        "ai_reasoning": ai_reasoning,
        "reference_reasoning": reference_reasoning
    }

In [38]:
#Confidence Calibration
def risk_band(score):
    if score < 0.4:
        return "low"
    elif score < 0.7:
        return "medium"
    else:
        return "high"

In [39]:
calibration_rows = []

for case in borderline_cases:
    ai_result = decision_agent(case)
    judge = llm_judge(case, ai_result)
    sem = semantic_evaluator(case, ai_result)

    calibration_rows.append({
        "case_id": case["case_id"],
        "domain": case["domain"],
        "ground_truth": case["ground_truth_decision"],
        "ai_decision": ai_result.get("decision"),
        "risk_score": ai_result.get("risk_score"),
        "risk_band": risk_band(ai_result.get("risk_score")),
        "judge_overall": judge.get("overall_score"),
        "semantic_similarity": sem.get("semantic_similarity"),
        "needs_human_review": ai_result.get("risk_score") >= 0.65
    })

calibration_df = pd.DataFrame(calibration_rows)
calibration_df

,case_id,domain,ground_truth,ai_decision,risk_score,risk_band,judge_overall,semantic_similarity,needs_human_review
0,B-001,ecommerce,escalate,escalate,0.7,high,9,0.662400,True
1,B-002,ecommerce,escalate,escalate,0.7,high,9,0.605548,True
2,B-003,finance,escalate,escalate,0.7,high,9,0.560805,True
3,B-004,finance,escalate,escalate,0.7,high,9,0.610073,True


## Final Analysis

### 1. Decision Accuracy
The model achieved high accuracy on synthetic cases, indicating strong alignment with policy rules.

### 2. Consistency
Even with temperature=0.3, decisions remained stable across runs.
Risk scores showed slight variance in some cases, indicating minor stochastic behavior.

### 3. Semantic Evaluation
Semantic similarity scores indicate that the model's reasoning is generally aligned with policy-based reference reasoning.
However, reasoning depth can be further improved.

### 4. Calibration
Risk scores are clustered (mostly around 0.7), suggesting limited calibration granularity.
Thresholding was introduced to map scores into operational actions.

### 5. Key Insight
The model performs reliably on structured policy-driven cases, but:
- Risk score calibration is coarse
- Reasoning depth can be improved
- Borderline cases require better differentiation

In [41]:
final_df = calibration_df.copy()

final_df = final_df[[
    "case_id",
    "domain",
    "ground_truth",
    "ai_decision",
    "risk_score",
    "risk_band",
    "judge_overall",
    "semantic_similarity",
    "needs_human_review"
]]

final_df

,case_id,domain,ground_truth,ai_decision,risk_score,risk_band,judge_overall,semantic_similarity,needs_human_review
0,B-001,ecommerce,escalate,escalate,0.7,high,9,0.662400,True
1,B-002,ecommerce,escalate,escalate,0.7,high,9,0.605548,True
2,B-003,finance,escalate,escalate,0.7,high,9,0.560805,True
3,B-004,finance,escalate,escalate,0.7,high,9,0.610073,True
